# HerBERT fine-tuning na PolEmo2 (Colab, T4 GPU)

1. Runtime → Change runtime type → **T4 GPU**
2. Opcjonalnie: dodaj `HF_TOKEN` w 🔑 Secrets (szybsze pobieranie z Hubu)
3. Uruchom komórki po kolei. Każdy trening ma osobną komórkę, po nim komórka z wynikami.

| Komórka | Tryb | Zadanie |
|---|---|---|
| 5 | LoRA (PEFT) | klasyfikacja tekstu |
| 6 | bez PEFT (tylko głowa) | klasyfikacja tekstu |
| 7 | LoRA (PEFT) | klasyfikacja tokenów |
| 8 | bez PEFT (tylko głowa) | klasyfikacja tokenów |
| 9 | podsumowanie wyników | — |

In [ ]:
REPO = "https://github.com/mtorzynska/nlp_language_models_lab.git"
BRANCH = "feature/herbert-finetuning"

!git clone -b $BRANCH $REPO
%cd nlp_language_models_lab

In [ ]:
# Colab ma już torch z CUDA — nie używaj uv sync, doinstaluj tylko brakujące pakiety
!pip install -q datasets transformers peft evaluate scikit-learn loguru matplotlib pyarrow

In [ ]:
# Opcjonalnie: token HF z sekretów Colab (unika throttlingu przy pobieraniu modelu)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN nie ustawiony — kontynuuję anonimowo")

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'BRAK — zmień runtime na T4'}")

In [ ]:
# Przygotowanie danych (parquet są w .gitignore, generujemy na miejscu)
!python -m src.data.prepare

## 1. Klasyfikacja tekstu — LoRA (PEFT)
Adaptery PEFT na wagach atencji (query/value), głowa klasyfikacyjna trenowana w całości.
Wyniki: `data/HerbertForSequenceClassification/results/metrics.json`.

In [ ]:
# ~15-20 min na T4. Na próbę: dodaj --max-train 1000 --epochs 1
!python -m src.models.train --task text --mode lora

### Wyniki — LoRA (tekst)

In [ ]:
from IPython.display import Image, display
import json

metrics = json.load(open("data/HerbertForSequenceClassification/results/metrics.json"))
print(json.dumps(metrics, indent=2))
display(Image(filename="data/HerbertForSequenceClassification/results/confusion_matrix.png"))

## 2. Klasyfikacja tekstu — bez PEFT (tylko głowa klasyfikacyjna)
Cały backbone zamrożony, trenowana tylko głowa. Punkt odniesienia dla LoRA.
⚠️ Nadpisze wyniki poprzedniego przebiegu (ten sam katalog).

In [ ]:
!python -m src.models.train --task text --mode head

### Wyniki — head (tekst)

In [ ]:
metrics = json.load(open("data/HerbertForSequenceClassification/results/metrics.json"))
print(json.dumps(metrics, indent=2))
display(Image(filename="data/HerbertForSequenceClassification/results/confusion_matrix.png"))

## 3. Klasyfikacja tokenów — LoRA (PEFT)
Etykiety na poziomie tokenów (pierwszy subtoken każdego słowa, `label_all_subwords=True`).
Wyniki: `data/HerbertForTokenClassification/results/metrics.json`.

In [ ]:
!python -m src.models.train --task token --mode lora

### Wyniki — LoRA (tokeny)

In [ ]:
metrics = json.load(open("data/HerbertForTokenClassification/results/metrics.json"))
print(json.dumps(metrics, indent=2))
display(Image(filename="data/HerbertForTokenClassification/results/confusion_matrix.png"))

## 4. Klasyfikacja tokenów — bez PEFT (tylko głowa)
⚠️ Nadpisze wyniki poprzedniego przebiegu (ten sam katalog).

In [ ]:
!python -m src.models.train --task token --mode head

### Wyniki — head (tokeny)

In [ ]:
metrics = json.load(open("data/HerbertForTokenClassification/results/metrics.json"))
print(json.dumps(metrics, indent=2))
display(Image(filename="data/HerbertForTokenClassification/results/confusion_matrix.png"))

## 5. Podsumowanie wszystkich przebiegów

In [ ]:
# Uruchom po każdym treningu, żeby zebrać wyniki w jednej tabeli.
import json
from pathlib import Path

runs = [
    ("text", "lora"),
    ("text", "head"),
    ("token", "lora"),
    ("token", "head"),
]

print(f"{'task':<6} {'mode':<5} {'f1 (macro)':>10} {'accuracy':>9}")
for task, mode in runs:
    path = Path(f"data/HerbertFor{'Sequence' if task == 'text' else 'Token'}Classification/results/metrics.json")
    if path.exists():
        m = json.loads(path.read_text())
        print(f"{task:<6} {mode:<5} {m['f1_score']:>10.4f} {m['accuracy']:>9.4f}")
    else:
        print(f"{task:<6} {mode:<5} {'—':>10} {'—':>9}")

## (Opcjonalnie) Zachowanie checkpointów w Drive
Sesje Colab potrafią się rozłączyć — checkpointy z końca każdej epoki są w `data/*/results/`.
Skopiuj je na Drive, żeby przetrwały po zamknięciu sesji.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lab2_results
!cp -r data/HerbertForSequenceClassification /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak wyników (tekst)'
!cp -r data/HerbertForTokenClassification /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak wyników (tokeny)'
!cp -r models /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak zapisanego modelu'